# VITS — Variational Inference with adversarial learning for end-to-end TTS

**Paper:** *Conditional Variational Autoencoder with Adversarial Learning for End-to-End Text-to-Speech* (Kim et al., 2021)
arXiv: [2106.06103](https://arxiv.org/abs/2106.06103)

---

## Motivation

Previous TTS pipelines (Tacotron 2, FastSpeech 2) have a **two-stage** design:

```
Text -> [Acoustic Model] -> Mel Spectrogram -> [Vocoder] -> Waveform
```

Problems with two-stage:
- Vocoder (e.g. WaveGlow, HiFi-GAN) must be trained separately
- Mel is an information bottleneck — fine details of waveform are lost
- Cascaded errors between stages

**VITS** is end-to-end: it learns to generate **raw waveform** directly from text, in a single model.

### Architecture Overview

```
Text (phonemes)
     |
 Text Encoder (Transformer)
     |
 Prior Distribution  N(mu_p, sigma_p)
     |   ^
     |   | KL divergence
     v   |
 Posterior Encoder (from mel/linear) -> N(mu_q, sigma_q)
     |
     z (sampled latent)
     |
 Normalizing Flow (RealNVP-style)
     |
 HiFi-GAN-style Decoder
     |
 Waveform
```

Training uses: **ELBO (reconstruction + KL) + GAN discriminator loss**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. Text Encoder

The text encoder maps phoneme IDs to a prior distribution `N(mu, sigma)`.
Architecture: **Transformer encoder** (multi-head attention + FFN) followed by a projection to 2*d_latent (mu and log-variance).

The prior is **conditional on text** — different from a simple standard normal prior.

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=192, n_heads=2, n_layers=6,
                 d_ff=768, d_latent=192, dropout=0.1):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = nn.Embedding(2000, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.proj = nn.Conv1d(d_model, d_latent * 2, kernel_size=1)  # -> mu, log_sigma

    def forward(self, x, src_key_padding_mask=None):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.embed(x) + self.pos_enc(pos)
        h = self.transformer(h, src_key_padding_mask=src_key_padding_mask)
        stats = self.proj(h.transpose(1, 2))       # (B, 2*d_latent, T)
        mu, log_sigma = stats.chunk(2, dim=1)       # each (B, d_latent, T)
        return mu, log_sigma

text_enc = TextEncoder(vocab_size=100, d_model=64, n_heads=2, n_layers=2, d_ff=256, d_latent=64)
phonemes = torch.randint(1, 100, (2, 15))
mu_p, log_sigma_p = text_enc(phonemes)
print("Prior mu:        ", mu_p.shape)
print("Prior log_sigma: ", log_sigma_p.shape)

## 2. Posterior Encoder

The posterior encoder takes the **mel spectrogram** (ground truth audio features) and produces `N(mu_q, sigma_q)`.

During training: we sample `z` from this posterior.
During inference: there is no audio, so we sample from the **prior** (text encoder output).

Architecture: WaveNet-style non-causal dilated convolutions.

In [ ]:
class PosteriorEncoder(nn.Module):
    def __init__(self, n_mels=80, d_hidden=192, d_latent=192, n_layers=16):
        super().__init__()
        self.pre  = nn.Conv1d(n_mels, d_hidden, kernel_size=1)
        convs = []
        for i in range(n_layers):
            dilation = 2 ** (i % 4)
            convs.append(nn.Conv1d(d_hidden, d_hidden * 2,
                                   kernel_size=5, dilation=dilation,
                                   padding=2 * dilation))
        self.convs = nn.ModuleList(convs)
        self.post  = nn.Conv1d(d_hidden, d_latent * 2, kernel_size=1)

    def forward(self, mel):
        # mel: (B, n_mels, T_mel)
        x = self.pre(mel)
        for conv in self.convs:
            h = conv(x)
            h_val, h_gate = h.chunk(2, dim=1)
            x = x + torch.tanh(h_val) * torch.sigmoid(h_gate)
        stats = self.post(x)                       # (B, 2*d_latent, T)
        mu, log_sigma = stats.chunk(2, dim=1)
        return mu, log_sigma

    def reparameterize(self, mu, log_sigma):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(log_sigma)

post_enc = PosteriorEncoder(n_mels=80, d_hidden=64, d_latent=64, n_layers=4)
mel = torch.randn(2, 80, 50)       # (B, n_mels, T_mel)
mu_q, log_sigma_q = post_enc(mel)
z = post_enc.reparameterize(mu_q, log_sigma_q)
print("Posterior mu:  ", mu_q.shape)
print("Sampled z:     ", z.shape)

## 3. KL Divergence Loss

VITS trains with the **Evidence Lower BOund (ELBO)**:

```
ELBO = E_q[log p(x|z)] - KL(q(z|x) || p(z|c))
```

where:
- `p(x|z)` = decoder reconstruction
- `q(z|x)` = posterior (from audio)
- `p(z|c)` = prior (from text)

The KL term forces the posterior distribution (from audio) to match the prior distribution (from text), so that at inference time we can sample from the text-conditioned prior.

In [ ]:
def kl_divergence_diagonal_gaussians(mu_q, log_sigma_q, mu_p, log_sigma_p):
    # KL(N(mu_q, sigma_q) || N(mu_p, sigma_p))
    # Closed-form: 0.5 * sum(sigma_q^2/sigma_p^2 + (mu_q-mu_p)^2/sigma_p^2 - 1 + log(sigma_p^2/sigma_q^2))
    var_q = torch.exp(2 * log_sigma_q)
    var_p = torch.exp(2 * log_sigma_p)
    kl = 0.5 * (var_q / var_p
                + (mu_q - mu_p).pow(2) / var_p
                - 1
                + 2 * (log_sigma_p - log_sigma_q))
    return kl.mean()

# Demo: prior from text encoder, posterior from audio encoder
# Align lengths (in practice done via MAS — Monotonic Alignment Search)
T_min = min(mu_p.size(2), mu_q.size(2))
kl = kl_divergence_diagonal_gaussians(
    mu_q[:, :, :T_min], log_sigma_q[:, :, :T_min],
    mu_p[:, :, :T_min], log_sigma_p[:, :, :T_min]
)
print(f"KL loss: {kl.item():.4f}")
print("When KL -> 0: posterior and prior distributions match")
print("This means at inference we can sample from prior (text only)")

## 4. Normalizing Flow

VITS uses a **normalizing flow** between the latent `z` and the decoder input.
The flow learns an invertible transformation, making the prior distribution more flexible.

During training:  `z_q -> Flow -> z_p` (posterior to prior space)
During inference: `z_p -> Flow^{-1} -> z_q` (prior to decoder space)

VITS uses a **Rational Quadratic Spline** coupling flow. Here we implement a simpler affine coupling layer for illustration.

In [ ]:
class AffineCouplingLayer(nn.Module):
    def __init__(self, d_latent, d_hidden=64):
        super().__init__()
        half = d_latent // 2
        self.net = nn.Sequential(
            nn.Conv1d(half, d_hidden, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(d_hidden, d_hidden, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(d_hidden, half * 2, kernel_size=1),  # -> scale, shift
        )

    def forward(self, x, reverse=False):
        # x: (B, d_latent, T)
        x0, x1 = x.chunk(2, dim=1)
        stats = self.net(x0)
        log_s, t = stats.chunk(2, dim=1)
        log_s = torch.tanh(log_s)           # bound log-scale for stability
        if not reverse:
            y1 = x1 * torch.exp(log_s) + t
            log_det = log_s.sum()
            return torch.cat([x0, y1], dim=1), log_det
        else:
            y1 = (x1 - t) * torch.exp(-log_s)
            return torch.cat([x0, y1], dim=1)

class NormalizingFlow(nn.Module):
    def __init__(self, d_latent=64, n_flows=4):
        super().__init__()
        self.flows = nn.ModuleList([AffineCouplingLayer(d_latent) for _ in range(n_flows)])

    def forward(self, z, reverse=False):
        log_det_total = 0
        if not reverse:
            for flow in self.flows:
                z, log_det = flow(z)
                log_det_total += log_det
            return z, log_det_total
        else:
            for flow in reversed(self.flows):
                z = flow(z, reverse=True)
            return z

flow = NormalizingFlow(d_latent=64, n_flows=4)
z_q = torch.randn(2, 64, 50)
z_p, log_det = flow(z_q)
z_q_recovered = flow(z_p, reverse=True)
print("z_q shape:          ", z_q.shape)
print("z_p shape:          ", z_p.shape)
print("log_det:            ", log_det.item())
print("Reconstruction err: ", (z_q - z_q_recovered).abs().max().item())

## 5. HiFi-GAN-style Decoder (Generator)

The decoder maps latent `z` directly to a **raw waveform** (not mel spectrogram).
It uses **multi-receptive field fusion (MRF)**: parallel residual blocks with different kernel sizes and dilation rates.

This is the same architecture as **HiFi-GAN v1**, adapted inside VITS.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilations=(1, 3, 5)):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.LeakyReLU(0.1),
                nn.Conv1d(channels, channels, kernel_size,
                          dilation=d, padding=d * (kernel_size - 1) // 2),
                nn.LeakyReLU(0.1),
                nn.Conv1d(channels, channels, kernel_size,
                          dilation=1, padding=(kernel_size - 1) // 2)
            ) for d in dilations
        ])
        self.skips = nn.ModuleList([
            nn.Conv1d(channels, channels, 1) for _ in dilations
        ])

    def forward(self, x):
        for conv, skip in zip(self.convs, self.skips):
            x = x + conv(skip(x))
        return x

class VITSDecoder(nn.Module):
    def __init__(self, d_latent=64, upsample_rates=(8, 8, 2, 2),
                 base_channels=512, mrf_kernels=(3, 7, 11)):
        super().__init__()
        self.pre = nn.Conv1d(d_latent, base_channels, kernel_size=7, padding=3)
        self.ups = nn.ModuleList()
        self.mrfs = nn.ModuleList()
        ch = base_channels
        for r in upsample_rates:
            self.ups.append(nn.ConvTranspose1d(ch, ch // 2,
                                                kernel_size=r * 2, stride=r,
                                                padding=r // 2))
            self.mrfs.append(nn.ModuleList([
                ResBlock(ch // 2, k) for k in mrf_kernels
            ]))
            ch = ch // 2
        self.post = nn.Sequential(
            nn.LeakyReLU(0.1),
            nn.Conv1d(ch, 1, kernel_size=7, padding=3),
            nn.Tanh()
        )

    def forward(self, z):
        # z: (B, d_latent, T_latent)
        x = self.pre(z)
        for up, mrf in zip(self.ups, self.mrfs):
            x = F.leaky_relu(up(x), 0.1)
            # Multi-receptive field fusion: average across parallel resblocks
            x = sum(rb(x) for rb in mrf) / len(mrf)
        return self.post(x)   # (B, 1, T_audio)

decoder = VITSDecoder(d_latent=64, base_channels=128, upsample_rates=(4, 4, 2, 2))
z_latent = torch.randn(1, 64, 20)
waveform  = decoder(z_latent)
print("Latent shape:  ", z_latent.shape)
print("Waveform shape:", waveform.shape)
upsample_factor = 4 * 4 * 2 * 2
print(f"Upsample factor: {upsample_factor}x  ({20 * upsample_factor} samples @ 22kHz = {20*upsample_factor/22050*1000:.1f} ms)")

## 6. Stochastic Duration Predictor

Unlike FastSpeech 2 (which uses a deterministic duration predictor), VITS uses a **stochastic duration predictor** — a normalizing flow that models the distribution over durations.

This makes duration prediction **probabilistic**, enabling natural variation in speaking rhythm.

During training: computes exact log-likelihood of ground-truth durations.
During inference: samples from the duration distribution.

In [ ]:
class StochasticDurationPredictor(nn.Module):
    def __init__(self, d_model, d_hidden=64, n_flows=4):
        super().__init__()
        self.pre  = nn.Conv1d(d_model, d_hidden, kernel_size=1)
        self.flows = nn.ModuleList([AffineCouplingLayer(d_hidden) for _ in range(n_flows)])
        self.post = nn.Conv1d(d_hidden, 2, kernel_size=1)   # mu, log_sigma of duration

    def forward(self, x, durations=None):
        # x: (B, d_model, T_text)
        h = self.pre(x)
        log_det = 0
        for flow in self.flows:
            h, ld = flow(h)
            log_det += ld
        stats = self.post(h)
        mu, log_sigma = stats[:, 0], stats[:, 1]
        if durations is not None:
            # NLL loss for training
            log_dur = torch.log(durations.float().clamp(min=1))
            nll = 0.5 * ((log_dur - mu).pow(2) * torch.exp(-2 * log_sigma) + 2 * log_sigma)
            return nll.mean() - log_det / x.numel()
        else:
            # Sample durations for inference
            eps = torch.randn_like(mu)
            log_dur = mu + eps * torch.exp(log_sigma)
            return torch.clamp(log_dur.exp().round().long(), min=1)

sdp = StochasticDurationPredictor(d_model=64)
enc_hidden = torch.randn(2, 64, 12)
gt_durs    = torch.randint(1, 6, (2, 12))
nll = sdp(enc_hidden, gt_durs)
print(f"Duration NLL (training): {nll.item():.4f}")
pred_durs = sdp(enc_hidden)
print("Sampled durations:", pred_durs[0].tolist())

## 7. Full VITS Pipeline

Putting all components together and showing the training vs. inference path.

In [ ]:
class VITS(nn.Module):
    def __init__(self, vocab_size=100, d_model=64, d_latent=64, n_mels=80):
        super().__init__()
        self.text_encoder = TextEncoder(vocab_size, d_model=d_model, n_heads=2,
                                         n_layers=2, d_ff=256, d_latent=d_latent)
        self.post_encoder = PosteriorEncoder(n_mels=n_mels, d_hidden=d_latent,
                                              d_latent=d_latent, n_layers=4)
        self.flow    = NormalizingFlow(d_latent=d_latent, n_flows=2)
        self.decoder = VITSDecoder(d_latent=d_latent, base_channels=64,
                                    upsample_rates=(4, 4, 2, 2))
        self.sdp     = StochasticDurationPredictor(d_model=d_model)

    def forward(self, phoneme_ids, mel=None):
        # Text encoder -> prior
        mu_p, log_sigma_p = self.text_encoder(phoneme_ids)      # (B, d_latent, T_text)

        if mel is not None:
            # Training: encode audio -> posterior -> sample z
            mu_q, log_sigma_q = self.post_encoder(mel)           # (B, d_latent, T_mel)
            z = self.post_encoder.reparameterize(mu_q, log_sigma_q)

            # Flow: map z_q -> z_p for KL computation
            T = min(mu_p.size(2), z.size(2))
            z_p, _ = self.flow(z[:, :, :T])

            kl = kl_divergence_diagonal_gaussians(
                z_p, log_sigma_q[:, :, :T],
                mu_p[:, :, :T], log_sigma_p[:, :, :T]
            )
            waveform = self.decoder(z)
            return waveform, kl
        else:
            # Inference: sample from prior, flow backward
            z_p = mu_p + torch.randn_like(mu_p) * torch.exp(log_sigma_p)
            z   = self.flow(z_p, reverse=True)
            waveform = self.decoder(z)
            return waveform

model_vits = VITS(vocab_size=100)

# Training forward pass
phonemes = torch.randint(1, 100, (2, 12))
mel_gt   = torch.randn(2, 80, 50)
wav, kl  = model_vits(phonemes, mel=mel_gt)
print("Training output:")
print("  Waveform:", wav.shape)
print(f"  KL loss: {kl.item():.4f}")

# Inference forward pass
model_vits.eval()
with torch.no_grad():
    wav_inf = model_vits(phonemes[:1])
    print("Inference output:")
    print("  Waveform:", wav_inf.shape)
print(f"Total parameters: {sum(p.numel() for p in model_vits.parameters()):,}")

## 8. Architecture Comparison: TTS Models

| Feature | WaveNet | Tacotron 2 | FastSpeech 2 | VITS |
|---------|---------|-----------|-------------|------|
| Autoregressive | Yes | Yes | No | No |
| End-to-end | No | No | No | Yes |
| Alignment | N/A | Attention | Duration predictor | MAS + SDP |
| Vocoder needed | Is a vocoder | Yes | Yes | No |
| Inference speed | Very slow | Slow | Fast | Fast |
| Prosody control | No | Limited | Explicit | Implicit (sampling) |
| Training signal | Waveform MLE | Mel MAE | Mel MAE + 3 losses | ELBO + GAN |
| Year | 2016 | 2018 | 2021 | 2021 |

### VITS Training Objective

```
L_total = L_recon + L_KL + L_dur + L_adv + L_fm
```

| Loss term | Description |
|-----------|-------------|
| L_recon | Mel spectrogram reconstruction (computed from generated waveform) |
| L_KL | KL divergence between posterior and prior |
| L_dur | Stochastic duration predictor NLL |
| L_adv | GAN adversarial loss (Multi-Period Discriminator) |
| L_fm | Feature matching loss (intermediate discriminator features) |

In [ ]:
# Visual comparison of TTS generation paradigms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Two-stage pipeline
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("Two-Stage (Tacotron2 + Vocoder)", fontsize=10)
boxes = [
    (1, 7, 3, 2, "Text", "#AED6F1"),
    (1, 4, 3, 2, "Acoustic Model\n(Tacotron 2)", "#85C1E9"),
    (1, 1, 3, 2, "Mel Spectrogram", "#AED6F1"),
    (6, 4, 3, 2, "Vocoder\n(WaveGlow)", "#F9E79F"),
    (6, 1, 3, 2, "Waveform", "#ABEBC6"),
]
for (x, y, w, h, label, color) in boxes:
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="k"))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=8)
ax.annotate("", xy=(2.5, 4+2), xytext=(2.5, 7), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(2.5, 1+2), xytext=(2.5, 4), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(6, 5), xytext=(4, 5), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(7.5, 1+2), xytext=(7.5, 4), arrowprops=dict(arrowstyle="->"))
ax.axis("off")

# 2. FastSpeech2 pipeline
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title("FastSpeech 2 + Vocoder", fontsize=10)
boxes2 = [
    (3.5, 10, 3, 1.5, "Text", "#AED6F1"),
    (3.5, 7.5, 3, 1.5, "Text Encoder\n(FFT)", "#85C1E9"),
    (3.5, 5, 3, 1.5, "Variance Adaptor\n(Dur/Pitch/Energy)", "#D7BDE2"),
    (3.5, 2.5, 3, 1.5, "Mel Decoder\n(FFT)", "#85C1E9"),
    (3.5, 0, 3, 1.5, "Mel -> Vocoder", "#F9E79F"),
]
for (x, y, w, h, label, color) in boxes2:
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="k"))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=7.5)
for y in [10, 7.5, 5, 2.5]:
    ax.annotate("", xy=(5, y), xytext=(5, y+1.5), arrowprops=dict(arrowstyle="->"))
ax.axis("off")

# 3. VITS (end-to-end)
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title("VITS (End-to-End)", fontsize=10)
boxes3 = [
    (3.5, 10, 3, 1.5, "Text", "#AED6F1"),
    (3.5, 7.5, 3, 1.5, "Text Encoder\n(Prior)", "#85C1E9"),
    (3.5, 5, 3, 1.5, "Normalizing Flow\n+ MAS", "#D7BDE2"),
    (3.5, 2.5, 3, 1.5, "Latent z", "#ABEBC6"),
    (3.5, 0, 3, 1.5, "HiFi-GAN Decoder\n(Waveform)", "#ABEBC6"),
]
for (x, y, w, h, label, color) in boxes3:
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="k"))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=7.5)
for y in [10, 7.5, 5, 2.5]:
    ax.annotate("", xy=(5, y), xytext=(5, y+1.5), arrowprops=dict(arrowstyle="->"))
ax.text(5, -0.5, "Single model, no separate vocoder",
        ha="center", fontsize=8, color="green", fontweight="bold")
ax.axis("off")

plt.suptitle("TTS Architecture Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/tts_comparison.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved figures/tts_comparison.png")

## Summary

| Component | Role |
|-----------|------|
| **Text Encoder** | Maps phonemes to prior distribution N(mu_p, sigma_p) |
| **Posterior Encoder** | Maps mel/audio to posterior N(mu_q, sigma_q) (training only) |
| **KL Divergence** | Forces posterior close to prior |
| **Normalizing Flow** | Flexible prior; invertible transform between z spaces |
| **SDP** | Stochastic duration prediction via flow |
| **HiFi-GAN Decoder** | Latent z -> raw waveform directly (no mel intermediate) |
| **GAN Discriminator** | Adversarial loss for realistic waveform quality |

**Key advantages of VITS:**
1. Single model — no separate vocoder to maintain
2. More natural prosody — stochastic duration + latent sampling
3. Competitive quality with FastSpeech 2 + HiFi-GAN, faster training pipeline

**Next:** Voice Cloning — adapting a TTS model to a new speaker from a few seconds of audio.